# Replace MELD `videoVisual` with Longformer VLM Embeddings
Encodes the `vlm_analysis` text from `meld_vlm_manifest.csv` using `allenai/longformer-base-4096`
(CLS token, no truncation) and saves a new pkl where `videoVisual` is replaced with 768-dim embeddings.

Key differences from IEMOCAP:
- PKL has **14 fields** (adds `videoSentiments` and a trailing `None`)
- Loaded **without** `encoding="latin1"`
- `videoVisual` dtype is **float64** in original → saved as **float32**
- Three-way split-aware key mapping (train / dev / test — Dialogue_IDs restart at 0 per split)

In [1]:
import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
PKL_IN     = "Dataset/CFN-ESA/meld_multi_features.pkl"
PKL_OUT    = "Dataset/CFN-ESA/meld_vlm_visual.pkl"
MANIFEST   = "meld_vlm_manifest.csv"
MODEL_NAME = "allenai/longformer-base-4096"
BATCH_SIZE = 8

# ── Split-offset constants (discovered in meld_dataset_analyzer) ──────────────
# PKL global_key → manifest Dialogue_ID:
#   0   .. 1038  →  train  Dialogue_ID = global_key
#   1039.. 1152  →  dev    Dialogue_ID = global_key - 1039
#   1153.. 1432  →  test   Dialogue_ID = global_key - 1153
N_TRAIN      = 1039
N_DEV_OFFSET = 1039
N_TEST_OFFSET = 1153   # = min(testVid)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/mnt/Work/Environments/Ubuntu/Conda/envs/ml/lib/python3.11/site-packages/torch/cuda/__init__.py:58: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device : cuda
GPU    : NVIDIA GeForce RTX 3060
VRAM   : 12.5 GB


/mnt/Work/Environments/Ubuntu/Conda/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Load the original PKL and the manifest

In [2]:
# ── Load original PKL ─────────────────────────────────────────────────────────
# MELD pkl is a list of 14 fields — no encoding="latin1" needed
(
    videoIDs, videoSpeakers, videoLabels, videoSentiments,
    videoText0, videoText1, videoText2, videoText3,
    videoAudio, videoVisual,
    videoSentence, trainVid, testVid, _extra,
) = pickle.load(open(PKL_IN, "rb"))

all_vids   = sorted(videoIDs.keys())
total_utts = sum(len(videoIDs[v]) for v in all_vids)
print(f"Dialogues loaded  : {len(all_vids)}")
print(f"Total utterances  : {total_utts}")
print(f"trainVid          : {len(trainVid)} dialogues")
print(f"testVid           : {len(testVid)}  dialogues")
print(f"Original visual dim : {np.array(videoVisual[all_vids[0]]).shape[1]}")
print(f"Original visual dtype: {np.array(videoVisual[all_vids[0]]).dtype}")

# ── Load manifest ─────────────────────────────────────────────────────────────
df = pd.read_csv(MANIFEST)
print(f"\nManifest rows : {len(df)}")
print(f"Columns       : {list(df.columns)}")

# Build per-split lookups: manifest_key → vlm_analysis text
def _build_lookup(df_split):
    return {
        f"dia{int(r['Dialogue_ID'])}_utt{int(r['Utterance_ID'])}": r['vlm_analysis']
        for _, r in df_split.iterrows()
    }

vlm_train = _build_lookup(df[df['split'] == 'train'])
vlm_dev   = _build_lookup(df[df['split'] == 'dev'])
vlm_test  = _build_lookup(df[df['split'] == 'test'])

print(f"\nvlm_train entries : {len(vlm_train)}")
print(f"vlm_dev   entries : {len(vlm_dev)}")
print(f"vlm_test  entries : {len(vlm_test)}")

Dialogues loaded  : 1432
Total utterances  : 13708
trainVid          : 1152 dialogues
testVid           : 280  dialogues
Original visual dim : 342
Original visual dtype: float64

Manifest rows : 13707
Columns       : ['sr_no', 'Utterance', 'Speaker', 'Emotion', 'Sentiment', 'Dialogue_ID', 'Utterance_ID', 'Season', 'Episode', 'StartTime', 'EndTime', 'split', 'path', 'vlm_analysis']

vlm_train entries : 9989
vlm_dev   entries : 1108
vlm_test  entries : 2610


## Step 2 — Coverage check: map every PKL utterance to its manifest VLM text

In [3]:
def get_vlm_text(dia_key, utt_id):
    """
    Return the VLM analysis text for a given PKL global dialogue key + utterance ID.
    Uses three-way split-aware offset mapping:
      global_key <  N_TRAIN      → train lookup  (no offset)
      global_key <  N_TEST_OFFSET → dev   lookup  (offset = N_DEV_OFFSET = 1039)
      global_key >= N_TEST_OFFSET → test  lookup  (offset = N_TEST_OFFSET = 1153)
    Returns empty string if not found (will produce zero-vector fallback).
    """
    if dia_key < N_TRAIN:
        key = f"dia{dia_key}_utt{utt_id}"
        return vlm_train.get(key, "")
    elif dia_key < N_TEST_OFFSET:
        key = f"dia{dia_key - N_DEV_OFFSET}_utt{utt_id}"
        return vlm_dev.get(key, "")
    else:
        key = f"dia{dia_key - N_TEST_OFFSET}_utt{utt_id}"
        return vlm_test.get(key, "")


# Build flat list of all utterances
flat_records = []   # (dia_key, utt_idx, utt_id)
flat_texts   = []   # parallel VLM analysis texts

for dia_key in all_vids:
    for utt_idx, utt_id in enumerate(videoIDs[dia_key]):
        text = get_vlm_text(dia_key, utt_id)
        flat_records.append((dia_key, utt_idx, utt_id))
        flat_texts.append(text)

# Coverage report
missing = [(flat_records[i], flat_texts[i]) for i in range(len(flat_texts)) if flat_texts[i] == ""]
found   = len(flat_texts) - len(missing)

print(f"PKL utterances          : {len(flat_texts)}")
print(f"Found in manifest       : {found}  ({100*found/len(flat_texts):.4f}%)")
print(f"Missing (empty fallback): {len(missing)}")
if missing:
    print("\nMissing entries → will get zero-vector (768-dim):")
    for rec, _ in missing:
        dia_key, utt_idx, utt_id = rec
        print(f"  dia_key={dia_key}  utt_idx={utt_idx}  utt_id={utt_id}")

PKL utterances          : 13708
Found in manifest       : 13707  (99.9927%)
Missing (empty fallback): 1

Missing entries → will get zero-vector (768-dim):
  dia_key=1149  utt_idx=7  utt_id=7


## Step 3 — Load Longformer

In [4]:
from transformers import AutoTokenizer, LongformerModel

print(f"Loading tokenizer and model : {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = LongformerModel.from_pretrained(MODEL_NAME, use_safetensors=True)
model.eval()
model.to(DEVICE)

hidden_size = model.config.hidden_size
print(f"Model hidden size (→ new visual dim)  : {hidden_size}")
print(f"Max position embeddings               : {model.config.max_position_embeddings}")
print("Model loaded successfully.")

Loading tokenizer and model : allenai/longformer-base-4096


Loading weights: 100%|██████████| 270/270 [00:00<00:00, 1264.67it/s, Materializing param=pooler.dense.weight]                                
LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                               | Status     | 
----------------------------------+------------+-
lm_head.bias                      | UNEXPECTED | 
lm_head.dense.bias                | UNEXPECTED | 
lm_head.decoder.weight            | UNEXPECTED | 
lm_head.layer_norm.bias           | UNEXPECTED | 
lm_head.dense.weight              | UNEXPECTED | 
lm_head.layer_norm.weight         | UNEXPECTED | 
embeddings.word_embeddings.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model hidden size (→ new visual dim)  : 768
Max position embeddings               : 4098
Model loaded successfully.


## Step 4 — Encode all utterances with Longformer (batched, CLS token)

In [5]:
def encode_texts_longformer(texts, tokenizer, model, device, batch_size=8):
    """
    Encode a list of texts using Longformer (CLS token, global attention at pos 0).
    Empty strings (missing VLM entries) produce zero vectors.
    Returns numpy array of shape (len(texts), hidden_size).
    """
    all_embeddings = []
    hidden_size    = model.config.hidden_size

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding batches", leave=False):
        batch_texts = texts[i : i + batch_size]

        # Replace empty strings with a neutral placeholder so tokenizer doesn't fail
        safe_texts = [t if t.strip() else "." for t in batch_texts]

        encoding = tokenizer(
            safe_texts,
            padding=True,
            truncation=False,   # Longformer handles up to 4096 tokens natively
            return_tensors="pt",
        )

        input_ids      = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)

        # Set global attention on CLS token (position 0) only
        global_attention_mask = torch.zeros_like(attention_mask)
        global_attention_mask[:, 0] = 1

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                global_attention_mask=global_attention_mask,
            )

        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().float().numpy()

        # Zero out embeddings that correspond to empty-string placeholders
        for j, orig_text in enumerate(batch_texts):
            if not orig_text.strip():
                cls_embeddings[j] = 0.0

        all_embeddings.append(cls_embeddings)

    return np.concatenate(all_embeddings, axis=0)  # (N, hidden_size)


# ── Token-length sanity check on a sample ────────────────────────────────────
print(f"Total utterances to encode : {len(flat_texts)}")
print(f"Batch size                 : {BATCH_SIZE}")
print(f"Number of batches          : {(len(flat_texts) + BATCH_SIZE - 1) // BATCH_SIZE}")
print()

sample_lengths = []
for t in flat_texts[:200]:
    if t.strip():
        ids = tokenizer(t, return_tensors="pt", truncation=False)["input_ids"]
        sample_lengths.append(ids.shape[1])

if sample_lengths:
    print(f"Token length stats (first 200 non-empty utterances):")
    print(f"  min={min(sample_lengths)}, max={max(sample_lengths)}, "
          f"mean={np.mean(sample_lengths):.0f}, median={np.median(sample_lengths):.0f}")
    print(f"  All within 4096 limit? {all(l <= 4096 for l in sample_lengths)}")

Total utterances to encode : 13708
Batch size                 : 8
Number of batches          : 1714

Token length stats (first 200 non-empty utterances):
  min=109, max=530, mean=219, median=208
  All within 4096 limit? True


In [6]:
# ── Run encoding ──────────────────────────────────────────────────────────────
print("Starting Longformer encoding...")
all_embeddings = encode_texts_longformer(
    flat_texts, tokenizer, model, DEVICE, batch_size=BATCH_SIZE
)
print(f"\nEncoding complete.")
print(f"Output shape : {all_embeddings.shape}  (expected: {len(flat_texts)} × {hidden_size})")
print(f"dtype        : {all_embeddings.dtype}")
print(f"value range  : [{all_embeddings.min():.4f}, {all_embeddings.max():.4f}]")

Starting Longformer encoding...


Encoding batches:   0%|          | 0/1714 [00:00<?, ?it/s]Input ids are automatically padded to be a multiple of `config.attention_window`: 512
                                                                     


Encoding complete.
Output shape : (13708, 768)  (expected: 13708 × 768)
dtype        : float32
value range  : [-2.8453, 12.1400]


## Step 5 — Rebuild `videoVisual` and save new PKL

In [7]:
# ── Rebuild videoVisual dict ──────────────────────────────────────────────────
# flat_records[i] = (dia_key, utt_idx, utt_id)
# all_embeddings[i] = 768-dim float32 CLS embedding

new_videoVisual = {dia_key: [None] * len(videoIDs[dia_key]) for dia_key in all_vids}

for i, (dia_key, utt_idx, utt_id) in enumerate(flat_records):
    new_videoVisual[dia_key][utt_idx] = all_embeddings[i]   # np.float32 (768,)

# Convert to numpy arrays (float32, matching text/audio feature dtype)
for dia_key in all_vids:
    new_videoVisual[dia_key] = np.array(new_videoVisual[dia_key], dtype=np.float32)

# Sanity check on sample dialogue
sample = all_vids[0]
arr_new = new_videoVisual[sample]
arr_old = np.array(videoVisual[sample])
print("new_videoVisual sanity check:")
print(f"  [{sample}]  new shape={arr_new.shape}  new dtype={arr_new.dtype}")
print(f"  [{sample}]  old shape={arr_old.shape}  old dtype={arr_old.dtype}")
assert arr_new.shape == (len(videoIDs[sample]), hidden_size), \
    f"Shape mismatch! got {arr_new.shape}"

# ── Reconstruct the 14-field list ─────────────────────────────────────────────
# Preserves all original fields, replacing only videoVisual (index 9).
# Field order matches MELDDataset_BERT unpacking:
#  0: videoIDs  1: videoSpeakers  2: videoLabels  3: videoSentiments
#  4: videoText0  5: videoText1  6: videoText2  7: videoText3
#  8: videoAudio  9: videoVisual  10: videoSentence  11: trainVid  12: testVid  13: None

new_data = [
    videoIDs, videoSpeakers, videoLabels, videoSentiments,
    videoText0, videoText1, videoText2, videoText3,
    videoAudio,
    new_videoVisual,     # ← replaced
    videoSentence, trainVid, testVid, _extra,
]

# ── Save ─────────────────────────────────────────────────────────────────────
Path(PKL_OUT).parent.mkdir(parents=True, exist_ok=True)
with open(PKL_OUT, "wb") as f:
    pickle.dump(new_data, f, protocol=4)

size_mb = Path(PKL_OUT).stat().st_size / 1e6
print(f"\nSaved to  : {PKL_OUT}")
print(f"File size : {size_mb:.1f} MB")
print("Done.")

new_videoVisual sanity check:
  [0]  new shape=(14, 768)  new dtype=float32
  [0]  old shape=(14, 342)  old dtype=float64

Saved to  : Dataset/CFN-ESA/meld_vlm_visual.pkl
File size : 286.9 MB
Done.


## Step 6 — Verify the new PKL by reloading from disk

In [8]:
# Reload from disk and run full assertions
print(f"Reloading : {PKL_OUT}")
(
    v_videoIDs, v_videoSpeakers, v_videoLabels, v_videoSentiments,
    v_videoText0, v_videoText1, v_videoText2, v_videoText3,
    v_videoAudio, v_videoVisual_new,
    v_videoSentence, v_trainVid, v_testVid, v_extra,
) = pickle.load(open(PKL_OUT, "rb"))

# ── Structural checks ─────────────────────────────────────────────────────────
assert len(v_videoIDs) == 1432,  f"Expected 1432 dialogues, got {len(v_videoIDs)}"
assert v_trainVid      == trainVid, "trainVid mismatch"
assert v_testVid       == testVid,  "testVid mismatch"

v_all_vids   = sorted(v_videoIDs.keys())
total_new    = sum(len(v_videoIDs[v]) for v in v_all_vids)
assert total_new == 13708, f"Expected 13708 utterances, got {total_new}"

# ── Visual dim check ──────────────────────────────────────────────────────────
for dia_key in v_all_vids:
    arr = v_videoVisual_new[dia_key]
    expected_len = len(v_videoIDs[dia_key])
    assert arr.shape == (expected_len, 768), \
        f"dia {dia_key}: expected ({expected_len}, 768), got {arr.shape}"
    assert arr.dtype == np.float32, \
        f"dia {dia_key}: expected float32, got {arr.dtype}"

# ── Other fields unchanged ────────────────────────────────────────────────────
assert np.allclose(
    np.array(v_videoText0[sample]), np.array(videoText0[sample])
), "videoText0 changed!"
assert np.allclose(
    np.array(v_videoAudio[sample]), np.array(videoAudio[sample])
), "videoAudio changed!"
assert v_videoLabels[sample]     == videoLabels[sample],     "videoLabels changed!"
assert v_videoSentiments[sample] == videoSentiments[sample], "videoSentiments changed!"

print("All assertions passed.\n")
print(f"  Dialogues      : {len(v_videoIDs)}")
print(f"  Utterances     : {total_new}")
print(f"  New visual dim : {v_videoVisual_new[all_vids[0]].shape[1]}  (was {np.array(videoVisual[all_vids[0]]).shape[1]})")
print(f"  New visual dtype: {v_videoVisual_new[all_vids[0]].dtype}  (was {np.array(videoVisual[all_vids[0]]).dtype})")
print(f"  Audio dim      : {np.array(v_videoAudio[all_vids[0]]).shape[1]}  (unchanged)")
print(f"  Text dim       : {np.array(v_videoText0[all_vids[0]]).shape[1]}  (unchanged)")
print(f"\nSample dialogue '{sample}':")
print(f"  old visual shape : {np.array(videoVisual[sample]).shape}")
print(f"  new visual shape : {v_videoVisual_new[sample].shape}")
print(f"\nPKL is ready for training → {PKL_OUT}")

Reloading : Dataset/CFN-ESA/meld_vlm_visual.pkl
All assertions passed.

  Dialogues      : 1432
  Utterances     : 13708
  New visual dim : 768  (was 342)
  New visual dtype: float32  (was float64)
  Audio dim      : 300  (unchanged)
  Text dim       : 1024  (unchanged)

Sample dialogue '0':
  old visual shape : (14, 342)
  new visual shape : (14, 768)

PKL is ready for training → Dataset/CFN-ESA/meld_vlm_visual.pkl
